Yanni 0530 updates:  
1. Map the indices into simple names and run the codes  
2. Do some simple analysis in the markdown
3. Use quarterly data to generate covariance matrices
4. Organize factor data(in org_data1.xlsx). For equity index, select PE and size(using TOTAL_ETF_ASSET_UNDER_MANAGEMENT as approximation); for bond index, select OAS,YTW, and TTM
  
questions:  
- why limiting T=100 and why do we use it to calculate omega_0 rather than the total time length?  
- How to solve the missing value problems? For convenient, I only use 10-year data here.
- why dropping TX60AR Index?
- How to introduce factor model/ BL method?


In [37]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# pip install scikit-learn
# pip install openpyxl
from sklearn.decomposition import PCA
from numpy.linalg import inv
from statsmodels.stats.outliers_influence import variance_inflation_factor
#from names import bbg2name # colm change 0

In [40]:
# As I don't have access to bbg api， i create the mapping dictionary by myself and save it in the org_data1 sheet  2
## Load and Preprocess Data from 2004-2024; from six Russell and Bloomberg Barclays bond indices
index_data = pd.read_excel("data/org_data1.xlsx",sheet_name = "Sheet1")
#index_data.drop('TX60AR Index',axis=1,inplace=True) # colm change 1
# clean up blanks
fixblanks={x:x.replace("\u202f"," ") for x in list(index_data.columns)}
#yanni 0530
bbg2name = {
    "AS51 Index"   : "AU S&P/ASX 200",
    "SX5E Index"   : "EMU Stoxx 50",
    "MXEF Index"   : "MSCI Emerging Mkts",
    "NKY Index"    : "Japan Nikkei 225",
    "UKX Index"    : "UK FTSE 100",
    "TX60AR Index" : "CA S&P/TSX 60",
    "SPX Index"    : "US S&P 500",
    "FNRET Index"  : "US REITs",
    "I05500CA Index": "Canada Gov Bond",
    "I05510CA Index": "Canada Corp Bond",
    "LEATTREU Index": "EMU Gov Bond",
    "LSG1TRGU Index": "UK Gov Bond",
    "I38292JP Index": "Japan Gov Bond",
    "I12877US Index": "US Long Treasury",
    "BEMUTRUU Index": "EM Gov Bond",
    "BATY0 Index"  : "AU Corp Bond",
    "LC61TRGU Index": "UK Corp Bond",
    "LECPTREU Index": "EMU Corp Bond",
    "LJC1TRJU Index": "Japan Corp Bond",
    "BACR0 Index"  : "US Corp Bond",
    "LF98TRUU Index": "US High Yield",
    "LUMSTRUU Index": "US MBS"
}
index_data=index_data.rename(columns=fixblanks).rename(columns=bbg2name) # colm 1
index_data=index_data.rename(columns=fixblanks)
list(bbg2name[x] if x in bbg2name.keys() else 'not found' for x in list(index_data.columns))
#ndex_data.columns.sort_values()
#sorted(list(bbg2name.keys()))
#set(index_data.columns)-set(bbg2name.keys())
index_data['Date'] = pd.to_datetime(index_data['Date'])
index_data.set_index('Date', inplace=True)
index_returns = index_data.fillna(method='ffill').pct_change().dropna() # colm change 2
returns_clean = index_returns.dropna()
index_returns.to_csv('data/org_data_copy.csv') # colm 3

# Save index_returns to a pickle file
index_returns.to_pickle('data/rename_index_returns.pkl') # Save to pickle


# Test the collinearity

vif_data = pd.DataFrame()
vif_data["Asset"] = returns_clean.columns
vif_data["VIF"] = [variance_inflation_factor(returns_clean.values, i) for i in range(returns_clean.shape[1])]

print(vif_data)


                 Asset        VIF
0       AU S&P/ASX 200   3.732610
1         EMU Stoxx 50   4.864236
2   MSCI Emerging Mkts   3.658077
3     Japan Nikkei 225   3.145635
4          UK FTSE 100   3.862953
5        CA S&P/TSX 60   5.909261
6           US S&P 500   7.090241
7             US REITs   2.855731
8      Canada Gov Bond  14.696270
9     Canada Corp Bond  17.221845
10        EMU Gov Bond   9.463091
11         UK Gov Bond  16.275047
12      Japan Gov Bond   4.165122
13    US Long Treasury  13.280447
14         EM Gov Bond  10.660188
15        AU Corp Bond  17.763181
16        UK Corp Bond  20.582933
17       EMU Corp Bond  15.071300
18     Japan Corp Bond   4.548434
19        US Corp Bond  15.672495
20       US High Yield   8.258134
21              US MBS   4.430280


/var/folders/0_/90gvz7w52k50nmdszkztsq4r0000gn/T/ipykernel_10615/2732642360.py:40: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  index_returns = index_data.fillna(method='ffill').pct_change().dropna() # colm change 2


The VIF results indicate that several fixed income indices especially UK Corp Bond (20.58), AU Corp Bond (17.76), and Canada Corp Bond (17.22) exhibit high multicollinearity, suggesting they are strongly explained by other assets in the dataset. In contrast, equity indices generally show lower VIFs (mostly < 7), indicating less redundancy and better diversification potential.  
Doe this result make sense?

In [41]:
index_returns
##Japan gov bond and EM gov Bond contain missing values in the first ten years

,AU S&P/ASX 200,EMU Stoxx 50,MSCI Emerging Mkts,Japan Nikkei 225,UK FTSE 100,CA S&P/TSX 60,US S&P 500,US REITs,Canada Gov Bond,Canada Corp Bond,...,Japan Gov Bond,US Long Treasury,EM Gov Bond,AU Corp Bond,UK Corp Bond,EMU Corp Bond,Japan Corp Bond,US Corp Bond,US High Yield,US MBS
Date,,,,,,,,,,,,,,,,,,,,,
2013-05-31,-0.050980,0.021254,-0.029410,-0.006228,0.023790,0.023974,0.020763,-0.070685,-0.016734,-0.008596,...,-0.012476,-0.019590,-0.030698,-0.004983,-0.015802,-0.001836,-0.002116,3.381166e-03,-0.005796,-0.015333
2013-06-28,-0.025165,-0.060315,-0.067947,-0.007058,-0.055843,-0.038677,-0.014999,-0.047333,-0.016485,-0.015793,...,0.000114,-0.042559,-0.049811,-0.012094,-0.042134,-0.016439,0.001991,-6.125269e-03,-0.026226,-0.009619
2013-07-31,0.051928,0.063614,0.007678,-0.000658,0.065255,0.030152,0.049462,0.019168,0.000913,0.005703,...,0.002704,0.009937,0.017394,0.006650,0.024263,0.008463,0.001959,1.053227e-02,0.018957,-0.000900
2013-08-30,0.016425,-0.016899,-0.019007,-0.020446,-0.031435,0.019394,-0.031298,-0.088858,-0.005223,-0.004433,...,0.004717,-0.013681,-0.025222,-0.005134,-0.007459,-0.002071,0.001921,1.299660e-07,-0.006079,-0.002859
2013-09-30,0.016343,0.063123,0.062310,0.079689,0.007686,0.011314,0.029749,0.017936,0.004299,0.003995,...,0.005439,0.017936,0.025851,0.005097,0.009473,0.006706,0.001686,6.109442e-03,0.009931,0.014079
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31,-0.032845,0.019062,-0.002865,0.044140,-0.013790,-0.033552,-0.024990,-0.065983,-0.000871,0.002528,...,-0.000871,-0.008321,-0.012175,0.003655,-0.005659,-0.003762,0.000151,7.094234e-03,-0.004264,-0.016472
2025-01-31,0.045735,0.079839,0.016634,-0.008073,0.061292,0.041861,0.027016,-0.003745,0.011782,0.009918,...,-0.006737,0.008775,0.015981,0.001405,0.011578,0.004425,-0.004311,4.373247e-03,0.013663,0.005108
2025-02-28,-0.042186,0.033417,0.003549,-0.061078,0.015654,-0.004066,-0.014242,0.037111,0.011082,0.007286,...,-0.007623,0.017629,0.015692,0.008999,0.004979,0.005963,-0.002543,7.915734e-03,0.006714,0.025497


In [47]:
def simulate_sensitivity_model(index_returns, shrinkage_lambda=0,annualization_factor = 12 , gamma = 0.01, n_sim = 1000):
    T = 100 # index_returns.shape[0]
    n_assets = index_returns.shape[1]

    # Estimated covariance (annualized)
    Sigma = index_returns.cov().values * annualization_factor 
    vol = np.sqrt(np.diag(Sigma))
    vol_df = pd.DataFrame({
        'Asset': index_returns.columns,
        'Volatility': vol
    })
    
    # Apply shrinkage if needed
    if shrinkage_lambda > 0:
        # shrink_target = np.identity(n_assets) * np.trace(Sigma) / n_assets
        shrink_target = np.diag(np.diag(Sigma))
        Sigma = (1 - shrinkage_lambda) * Sigma + shrinkage_lambda * shrink_target


    # Omega_0 = covariance of theta
    Omega_0 = Sigma / T
    theta_0 = np.zeros((n_assets, 1))  # mean zero
    return MLBsim(Sigma,Omega_0,theta_0,gamma,n_sim)

def MLBsim(Sigma,Omega_0,theta_0,gamma,n_sim):
    # M² = trace(Sigma⁻¹) / n
    n_assets = Sigma.shape[0]
    Sigma_inv = np.linalg.inv(Sigma)
    trace_Sigma_inv = np.trace(Sigma_inv)
    M = np.sqrt(trace_Sigma_inv / n_assets)

    # L = 1 because Γ = Omega is fixed
    L = 1.0

    # Simulate theta and compute h
    h_1_norm_list = []
    h_2_norm_list = []

    for _ in range(n_sim):
        theta = np.random.multivariate_normal(mean=np.zeros(n_assets), cov=Omega_0).reshape(-1, 1)
        h = gamma * Sigma_inv @ theta
        h_1 = np.sum(np.abs(h))
        h_2 = np.sum(h ** 2)
        h_1_norm_list.append(h_1)
        h_2_norm_list.append(h_2)

    # Compute expectation estimates
    E_l1_squared = (np.mean(h_1_norm_list)) ** 2
    E_l2 = np.mean(h_2_norm_list)

    B = E_l1_squared / E_l2
    S = M * L * np.sqrt(B)

    return { "M": M, "L": L,"B": B, "S": S}

def simulate_sensitivity_pca_model(index_returns,annualization_factor=12, n_factors=None, gamma=0.01, n_sim=1000):
    n_returns, n_assets = index_returns.shape
    T=100
    returns_matrix = index_returns.values
    len(np.argwhere(np.isnan(returns_matrix)))
    Sigma = np.cov(returns_matrix.T) * annualization_factor 
    theta_0 = np.zeros((n_assets, 1))  # mean zero
    # === Step 1: PCA Decomposition ===
    n_factors = min(n_factors, T, n_assets)
    eigenvalues, eigenvectors = np.linalg.eigh(Sigma)
    evecn=eigenvectors[:,-n_factors:]
    evaln=eigenvalues[-n_factors:]
    Sigma_pca=evecn@np.diag(evaln)@evecn.T
    np.fill_diagonal(Sigma_pca, np.diag(Sigma))
    explained_variance_ratio= np.sum(evaln) / np.sum(eigenvalues)  # Explained variance ratio
    Omega_0= Sigma_pca / T
    mlb= MLBsim(Sigma_pca,Omega_0,theta_0,gamma,n_sim)
     
    mlb.update({"explained_variance_ratio":explained_variance_ratio, "n_factors": n_factors})
    return mlb
    

def run_sensitivity_analysis(index_returns, gamma, shrinkage_vec,n_factors_vec,T,time_windows):
    results_fixed = []
    results_random = []
    for window in time_windows:
        end_date = index_returns.index.max()
        start_date = end_date - pd.DateOffset(years=window)
        data_slice = index_returns[start_date:]
        #test different shrinkage rates
        for i in np.arange(len(shrinkage_vec)):
            result_random = simulate_sensitivity_model(data_slice,  gamma=gamma, shrinkage_lambda=shrinkage_vec[i])
            result_random.update({"Shrinkage Lambda": shrinkage_vec[i],"Time Window (Years)": window})
            results_random.append(result_random)
        # test the main factor contribution
        for i in np.arange(len(n_factors_vec)):
            #yanni0530
            nf = n_factors_vec[i]
            if nf > min(T, data_slice.shape[1]):
                continue  # skip if n_factors exceeds limit
            result_fixed = simulate_sensitivity_pca_model(data_slice,  n_factors=n_factors_vec[i],gamma=gamma)
            result_fixed.update({"Time Window (Years)": window})
            results_fixed.append(result_fixed)         
    results_df_fixed = pd.DataFrame(results_fixed)
    results_df_random = pd.DataFrame(results_random)
    return results_df_fixed,results_df_random


In [48]:
# Reload index_returns from the pickle file
returns_clean = pd.read_pickle('data/rename_index_returns.pkl') # Reload from pickle
shrinkage_vec = [0, 0.2, 0.4, 0.6, 0.8]
n_factors_vec=np.arange(0,24,1)

time_windows = [10]
gamma = 0.01
annualization_factor=12
T = 100

# Run and display
r0,r1 = run_sensitivity_analysis(returns_clean, gamma, shrinkage_vec,n_factors_vec, T,time_windows)
r1.to_csv("result_diff_shrinkage_rates.csv")
r0.to_csv("result_PCA.csv")

In [49]:
print(r0.sort_values(by=["Time Window (Years)","explained_variance_ratio"]))


            M    L         B           S  explained_variance_ratio  n_factors  \
1   32.148388  1.0  7.473565   87.886688                  0.633150          1   
2   34.177109  1.0  8.081190   97.156753                  0.743164          2   
3   36.758046  1.0  8.630445  107.986384                  0.818248          3   
4   39.981421  1.0  9.161638  121.016557                  0.870039          4   
5   40.821474  1.0  8.967969  122.246306                  0.902676          5   
6   41.367509  1.0  9.296049  126.127147                  0.925330          6   
7   42.806403  1.0  9.671725  133.125332                  0.943760          7   
8   44.869706  1.0  9.922679  141.340851                  0.958343          8   
9   45.338059  1.0  9.838235  142.207175                  0.968719          9   
10  47.276923  1.0  9.875927  148.572397                  0.976914         10   
11  49.209678  1.0  9.883381  154.704628                  0.982969         11   
12  52.340921  1.0  9.895399

In [50]:
print(r1)

           M    L         B           S  Shrinkage Lambda  Time Window (Years)
0  82.391067  1.0  7.497869  225.605170               0.0                   10
1  43.156419  1.0  6.612781  110.978144               0.2                   10
2  35.763096  1.0  6.634526   92.117057               0.4                   10
3  32.068834  1.0  5.981083   78.428350               0.6                   10
4  29.927669  1.0  6.019888   73.428912               0.8                   10


## Analysis
- The sensitivity decomposition results after data adjustment align well with expectations. In the PCA approach, the first five principal components account for over 90% of the variance, and 1% of risk corresponds to an average active share of 122%, which is consistent with the asset allocation case discussed in Section 7.1 of the paper. The number of investment opportunities is around 7–9, and the average position sizes also reflect realistic behavior.
- In the shrinkage method, a shrinkage rate of 0.2 already delivers notably improved results.

## Quarterly Data


In [51]:
index_returns

,AU S&P/ASX 200,EMU Stoxx 50,MSCI Emerging Mkts,Japan Nikkei 225,UK FTSE 100,CA S&P/TSX 60,US S&P 500,US REITs,Canada Gov Bond,Canada Corp Bond,...,Japan Gov Bond,US Long Treasury,EM Gov Bond,AU Corp Bond,UK Corp Bond,EMU Corp Bond,Japan Corp Bond,US Corp Bond,US High Yield,US MBS
Date,,,,,,,,,,,,,,,,,,,,,
2013-05-31,-0.050980,0.021254,-0.029410,-0.006228,0.023790,0.023974,0.020763,-0.070685,-0.016734,-0.008596,...,-0.012476,-0.019590,-0.030698,-0.004983,-0.015802,-0.001836,-0.002116,3.381166e-03,-0.005796,-0.015333
2013-06-28,-0.025165,-0.060315,-0.067947,-0.007058,-0.055843,-0.038677,-0.014999,-0.047333,-0.016485,-0.015793,...,0.000114,-0.042559,-0.049811,-0.012094,-0.042134,-0.016439,0.001991,-6.125269e-03,-0.026226,-0.009619
2013-07-31,0.051928,0.063614,0.007678,-0.000658,0.065255,0.030152,0.049462,0.019168,0.000913,0.005703,...,0.002704,0.009937,0.017394,0.006650,0.024263,0.008463,0.001959,1.053227e-02,0.018957,-0.000900
2013-08-30,0.016425,-0.016899,-0.019007,-0.020446,-0.031435,0.019394,-0.031298,-0.088858,-0.005223,-0.004433,...,0.004717,-0.013681,-0.025222,-0.005134,-0.007459,-0.002071,0.001921,1.299660e-07,-0.006079,-0.002859
2013-09-30,0.016343,0.063123,0.062310,0.079689,0.007686,0.011314,0.029749,0.017936,0.004299,0.003995,...,0.005439,0.017936,0.025851,0.005097,0.009473,0.006706,0.001686,6.109442e-03,0.009931,0.014079
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31,-0.032845,0.019062,-0.002865,0.044140,-0.013790,-0.033552,-0.024990,-0.065983,-0.000871,0.002528,...,-0.000871,-0.008321,-0.012175,0.003655,-0.005659,-0.003762,0.000151,7.094234e-03,-0.004264,-0.016472
2025-01-31,0.045735,0.079839,0.016634,-0.008073,0.061292,0.041861,0.027016,-0.003745,0.011782,0.009918,...,-0.006737,0.008775,0.015981,0.001405,0.011578,0.004425,-0.004311,4.373247e-03,0.013663,0.005108
2025-02-28,-0.042186,0.033417,0.003549,-0.061078,0.015654,-0.004066,-0.014242,0.037111,0.011082,0.007286,...,-0.007623,0.017629,0.015692,0.008999,0.004979,0.005963,-0.002543,7.915734e-03,0.006714,0.025497


In [52]:
# Convert to quarterly returns using compound returns:
index_returns_q = (1 + index_returns).resample('Q').prod() - 1
index_returns_q

/var/folders/0_/90gvz7w52k50nmdszkztsq4r0000gn/T/ipykernel_10615/2665370101.py:2: FutureWarning: 'Q' is deprecated and will be removed in a future version, please use 'QE' instead.
  index_returns_q = (1 + index_returns).resample('Q').prod() - 1


,AU S&P/ASX 200,EMU Stoxx 50,MSCI Emerging Mkts,Japan Nikkei 225,UK FTSE 100,CA S&P/TSX 60,US S&P 500,US REITs,Canada Gov Bond,Canada Corp Bond,...,Japan Gov Bond,US Long Treasury,EM Gov Bond,AU Corp Bond,UK Corp Bond,EMU Corp Bond,Japan Corp Bond,US Corp Bond,US High Yield,US MBS
Date,,,,,,,,,,,,,,,,,,,,,
2013-06-30,-0.074863,-0.040343,-0.095358,-0.013242,-0.033382,-0.015630,0.005452,-0.114672,-0.032943,-0.024254,...,-0.012363,-0.061315,-0.078980,-0.017016,-0.057270,-0.018245,-0.000129,-0.002765,-0.031869,-0.024804
2013-09-30,0.086679,0.111643,0.050121,0.056918,0.039699,0.062012,0.046860,-0.054738,-0.000034,0.005244,...,0.012913,0.013987,0.017371,0.006586,0.026254,0.013123,0.005576,0.016706,0.022820,0.010270
2013-12-31,0.025549,0.074607,0.015423,0.126974,0.044392,0.076916,0.099200,0.010567,-0.003028,0.009327,...,0.001737,0.017383,0.016198,-0.002182,0.002211,0.009591,0.002316,0.008401,0.035817,-0.004174
2014-03-31,0.007962,0.016919,-0.008018,-0.089832,-0.022332,0.055181,0.012974,0.079599,0.023434,0.025862,...,0.007995,0.023264,0.037072,0.012260,0.023275,0.023597,0.003203,0.015186,0.029813,0.015854
2014-06-30,0.000170,0.021078,0.056432,0.022543,0.022062,0.063238,0.046941,0.054949,0.015413,0.015367,...,0.007587,0.040271,0.046102,0.032875,0.022569,0.023936,0.003268,0.024983,0.024077,0.024065
2014-09-30,-0.019077,-0.000716,-0.043254,0.066707,-0.017975,0.004304,0.006152,-0.030125,0.009315,0.006300,...,0.005841,-0.003063,-0.000553,0.009941,0.029299,0.018387,0.001599,0.011282,-0.018680,0.001823
2014-12-31,0.022333,-0.024644,-0.048760,0.078972,-0.008551,-0.003552,0.043913,0.112028,0.025523,0.014862,...,0.022848,-0.020833,-0.006212,0.044618,0.044641,0.015537,0.004678,0.026990,-0.010022,0.017864
2015-03-31,0.088798,0.175103,0.019094,0.100639,0.031518,0.024161,0.004366,0.051139,0.035026,0.031694,...,-0.005024,0.021312,0.021547,0.030800,0.034451,0.013597,-0.001789,0.022119,0.025219,0.010573
2015-06-30,-0.073410,-0.073858,-0.002381,0.053561,-0.037215,-0.017522,-0.002312,-0.131326,-0.015081,-0.008329,...,-0.002223,0.014400,-0.001146,-0.027091,-0.041852,-0.029006,0.001282,-0.006990,0.000030,-0.007442


In [54]:
shrinkage_vec = [0, 0.2, 0.4, 0.6, 0.8]
n_factors_vec=np.arange(0,24,1)

time_windows = [10]
gamma = 0.01
annualization_factor=4
T=100

# Run and display
r2,r3 = run_sensitivity_analysis(index_returns_q, gamma, shrinkage_vec,n_factors_vec, T, time_windows)


In [55]:
r2

,M,L,B,S,explained_variance_ratio,n_factors,Time Window (Years)
0,63.331207,1.0,8.139571,180.683510,1.000000,0,10
1,18.197513,1.0,7.695607,50.481640,0.673765,1,10
2,20.036234,1.0,8.639577,58.892819,0.797434,2,10
3,22.342999,1.0,9.274983,68.045284,0.874593,3,10
4,24.812261,1.0,9.714849,77.336474,0.909379,4,10
5,25.969218,1.0,9.338566,79.359508,0.934025,5,10
6,26.461019,1.0,9.605599,82.010372,0.950968,6,10
7,27.397109,1.0,10.066547,86.925059,0.964265,7,10
8,29.647732,1.0,10.534945,96.229365,0.973952,8,10
9,32.961390,1.0,10.002861,104.247976,0.981009,9,10


In [57]:
r3

,M,L,B,S,Shrinkage Lambda,Time Window (Years)
0,63.331207,1.0,8.056592,179.760168,0.0,10
1,25.028392,1.0,6.646026,64.522913,0.2,10
2,20.353863,1.0,6.300746,51.090817,0.4,10
3,18.101557,1.0,6.214337,45.124596,0.6,10
4,16.829072,1.0,5.858306,40.732980,0.8,10


## Alpha Factor Construction